# 🔧 Task 4: Feature Engineering

---

## 🎯 Objective
Create powerful new features based on EDA insights to improve model performance. Following Zinkevich's Rules #16-#22 for feature engineering.

## 📚 Learning Objectives
1. Understand polynomial and interaction features
2. Learn domain-specific feature creation
3. Apply feature selection techniques
4. Build reusable feature transformers

---

## 🔬 Mathematical Foundation: Feature Engineering

### 1. Polynomial Features

Transform input features to include polynomial terms for capturing non-linear relationships:

For single feature $x$, degree $d$:
$$\phi(x) = [1, x, x^2, x^3, ..., x^d]$$

For two features $x_1, x_2$, degree 2:
$$\phi(x_1, x_2) = [1, x_1, x_2, x_1^2, x_1 x_2, x_2^2]$$

**Why it works:** A linear model on polynomial features becomes non-linear in original space:
$$y = w_0 + w_1 x + w_2 x^2 \quad \text{(parabola)}$$

---

### 2. Interaction Features

Capture combined effects of two variables:
$$x_{interaction} = x_1 \times x_2$$

**Example:** Wind speed alone and payload alone may have small effects, but combined (windy + heavy payload) may have multiplicative effect on power.

---

### 3. Domain-Specific Transformations

**Log Transform:** For highly skewed data:
$$x_{log} = \log(x + 1)$$

**Square Root:** For count-like data:
$$x_{sqrt} = \sqrt{x}$$

**Binning:** Convert continuous to categorical:
$$x_{bin} = \text{bin}(x) \in \{low, medium, high\}$$

---

### 4. Physics-Based Features (Domain Knowledge)

**Available Energy:**
$$E_{available} = SoC \times Q_{battery} \times V_{nominal}$$

**Power-to-Weight Ratio:**
$$PWR = \frac{P}{m_{total}}$$

**Efficiency Factor:**
$$\eta = \frac{E_{theoretical}}{E_{actual}}$$

---

## 📦 Import Libraries & Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully!")

In [ ]:
# Load raw data
df = pd.read_csv('../data/raw/uav_telemetry.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} features")
print(f"\nOriginal features: {list(df.columns)}")

## 🔧 Step 1: Polynomial Feature Generator

In [ ]:
class PolynomialFeatures:
    """
    Generate polynomial features from scratch.
    
    For degree=2: [x1, x2] -> [x1, x2, x1^2, x1*x2, x2^2]
    """
    
    def __init__(self, degree=2, include_bias=False, interaction_only=False):
        """
        Parameters:
        -----------
        degree : int
            Maximum polynomial degree
        include_bias : bool
            Include a bias column (all 1s)
        interaction_only : bool
            Only include interaction terms (no x^2, x^3, etc.)
        """
        self.degree = degree
        self.include_bias = include_bias
        self.interaction_only = interaction_only
        self.feature_names = None
        self.n_input_features = None
    
    def _generate_combinations(self, n_features):
        """Generate all polynomial combinations up to degree."""
        from itertools import combinations_with_replacement
        
        combos = []
        feature_indices = list(range(n_features))
        
        for d in range(1, self.degree + 1):
            for combo in combinations_with_replacement(feature_indices, d):
                if self.interaction_only and len(set(combo)) != len(combo):
                    # Skip terms like x^2 (same feature repeated)
                    continue
                combos.append(combo)
        
        return combos
    
    def fit(self, X, feature_names=None):
        """
        Fit to determine output feature structure.
        """
        X = np.array(X)
        self.n_input_features = X.shape[1]
        
        if feature_names is None:
            feature_names = [f'x{i}' for i in range(self.n_input_features)]
        
        self.input_feature_names = feature_names
        self.combinations = self._generate_combinations(self.n_input_features)
        
        # Generate output feature names
        self.feature_names = []
        if self.include_bias:
            self.feature_names.append('1')
        
        for combo in self.combinations:
            if len(combo) == 1:
                self.feature_names.append(feature_names[combo[0]])
            else:
                terms = [feature_names[i] for i in combo]
                # Count occurrences for power notation
                from collections import Counter
                counts = Counter(terms)
                name_parts = []
                for term, count in sorted(counts.items()):
                    if count == 1:
                        name_parts.append(term)
                    else:
                        name_parts.append(f'{term}^{count}')
                self.feature_names.append('*'.join(name_parts))
        
        return self
    
    def transform(self, X):
        """
        Transform input features to polynomial features.
        """
        X = np.array(X)
        n_samples = X.shape[0]
        
        # Calculate output
        n_output_features = len(self.combinations)
        if self.include_bias:
            n_output_features += 1
        
        X_poly = np.empty((n_samples, n_output_features))
        
        col = 0
        if self.include_bias:
            X_poly[:, col] = 1
            col += 1
        
        for combo in self.combinations:
            X_poly[:, col] = np.prod([X[:, i] for i in combo], axis=0)
            col += 1
        
        return X_poly
    
    def fit_transform(self, X, feature_names=None):
        """Fit and transform in one step."""
        self.fit(X, feature_names)
        return self.transform(X)
    
    def get_feature_names(self):
        """Return feature names."""
        return self.feature_names


# Demonstrate polynomial features
sample = np.array([[2, 3], [4, 5]])
poly = PolynomialFeatures(degree=2)
sample_poly = poly.fit_transform(sample, ['a', 'b'])

print("Polynomial Features Example:")
print(f"Input: {sample[0]}")
print(f"Output: {sample_poly[0]}")
print(f"Feature names: {poly.get_feature_names()}")

## ⚡ Step 2: Physics-Based Feature Engineering

In [ ]:
class UAVFeatureEngineer:
    """
    Domain-specific feature engineering for UAV telemetry.
    
    Creates physics-based features that capture real-world relationships.
    """
    
    # UAV Constants
    BATTERY_CAPACITY_WH = 222  # 10000mAh * 22.2V / 1000
    UAV_MASS = 3.5  # kg
    GRAVITY = 9.81  # m/s²
    
    def __init__(self):
        self.feature_names = []
    
    def create_energy_features(self, df):
        """
        Create energy-related features.
        
        E = SoC × Battery_Capacity
        """
        features = pd.DataFrame()
        
        # Available energy (Wh)
        features['available_energy_wh'] = (df['battery_soc'] / 100) * self.BATTERY_CAPACITY_WH
        
        # Energy density (Wh per % SoC)
        features['energy_per_soc'] = features['available_energy_wh'] / (df['battery_soc'] + 0.1)
        
        # Theoretical flight time (hours)
        features['theoretical_flight_time'] = features['available_energy_wh'] / (df['power_consumption'] + 1)
        
        return features
    
    def create_efficiency_features(self, df):
        """
        Create efficiency-related features.
        """
        features = pd.DataFrame()
        
        # Power efficiency (distance per watt)
        features['range_per_watt'] = df['max_range'] / (df['power_consumption'] + 1)
        
        # Voltage efficiency (how close to nominal)
        nominal_voltage = 22.2
        features['voltage_efficiency'] = df['voltage'] / nominal_voltage
        
        # Current efficiency (lower is better at same power)
        features['current_ratio'] = df['current'] / (df['current'].max() + 0.1)
        
        return features
    
    def create_environmental_features(self, df):
        """
        Create environmental impact features.
        """
        features = pd.DataFrame()
        
        # Wind squared (quadratic relationship found in EDA)
        features['wind_speed_squared'] = df['wind_speed'] ** 2
        
        # Wind power impact
        features['wind_power_factor'] = 1 + 0.025 * features['wind_speed_squared']
        
        # Dust impact
        features['dust_impact'] = 1 + 0.005 * df['dust_level']
        
        # Combined environmental stress
        features['environmental_stress'] = features['wind_power_factor'] * features['dust_impact']
        
        # Temperature deviation from optimal (25°C)
        features['temp_deviation'] = np.abs(df['ambient_temperature'] - 25)
        features['temp_efficiency'] = 1 - 0.02 * features['temp_deviation']
        
        return features
    
    def create_weight_features(self, df):
        """
        Create weight and load-related features.
        """
        features = pd.DataFrame()
        
        # Total weight
        features['total_weight'] = self.UAV_MASS + df['payload_weight']
        
        # Load ratio
        features['load_ratio'] = df['payload_weight'] / self.UAV_MASS
        
        # Thrust-to-weight estimate
        features['thrust_requirement'] = features['total_weight'] * self.GRAVITY
        
        # Power loading (W per kg)
        features['power_loading'] = df['power_consumption'] / features['total_weight']
        
        return features
    
    def create_battery_health_features(self, df):
        """
        Create battery state features.
        """
        features = pd.DataFrame()
        
        # SoC categories
        features['soc_low'] = (df['battery_soc'] < 30).astype(int)
        features['soc_medium'] = ((df['battery_soc'] >= 30) & (df['battery_soc'] < 70)).astype(int)
        features['soc_high'] = (df['battery_soc'] >= 70).astype(int)
        
        # Battery temperature stress
        features['battery_temp_stress'] = np.abs(df['battery_temperature'] - 35)  # Optimal ~35°C
        
        # Voltage-SoC relationship (detect anomalies)
        expected_voltage = 18 + (df['battery_soc'] / 100) * (25.2 - 18)
        features['voltage_deviation'] = np.abs(df['voltage'] - expected_voltage)
        
        return features
    
    def create_interaction_features(self, df):
        """
        Create important interaction features.
        """
        features = pd.DataFrame()
        
        # SoC × Environmental stress
        features['soc_wind_interaction'] = df['battery_soc'] * df['wind_speed']
        
        # Payload × Wind (heavy + windy = bad)
        features['payload_wind_interaction'] = df['payload_weight'] * df['wind_speed']
        
        # SoC × Power (high SoC with low power = good)
        features['soc_power_ratio'] = df['battery_soc'] / (df['power_consumption'] / 100 + 1)
        
        # Speed × Altitude
        features['speed_altitude_interaction'] = df['flight_speed'] * df['altitude'] / 100
        
        return features
    
    def fit_transform(self, df):
        """
        Apply all feature engineering transformations.
        """
        all_features = pd.DataFrame(index=df.index)
        
        # Generate all feature groups
        energy_features = self.create_energy_features(df)
        efficiency_features = self.create_efficiency_features(df)
        environmental_features = self.create_environmental_features(df)
        weight_features = self.create_weight_features(df)
        battery_features = self.create_battery_health_features(df)
        interaction_features = self.create_interaction_features(df)
        
        # Combine all features
        all_features = pd.concat([
            energy_features,
            efficiency_features,
            environmental_features,
            weight_features,
            battery_features,
            interaction_features
        ], axis=1)
        
        self.feature_names = list(all_features.columns)
        
        return all_features


print("✅ UAVFeatureEngineer class defined!")

In [ ]:
# Apply feature engineering
engineer = UAVFeatureEngineer()
engineered_features = engineer.fit_transform(df)

print(f"\nGenerated {len(engineered_features.columns)} new features:")
print("=" * 50)
for i, name in enumerate(engineered_features.columns, 1):
    print(f"  {i:2d}. {name}")

print("\n" + "=" * 50)
print("Feature Statistics:")
engineered_features.describe().round(2)

## 📊 Step 3: Visualize Engineered Features

In [ ]:
# Combine original and engineered features for analysis
df_combined = pd.concat([df, engineered_features], axis=1)

# Visualize key engineered features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 1. Available Energy vs Max Range
ax = axes[0, 0]
ax.scatter(df_combined['available_energy_wh'], df_combined['max_range'], 
           alpha=0.3, s=10, c='blue')
ax.set_xlabel('Available Energy (Wh)')
ax.set_ylabel('Max Range (km)')
ax.set_title('Available Energy vs Range', fontweight='bold')

# 2. Environmental Stress vs Power
ax = axes[0, 1]
ax.scatter(df_combined['environmental_stress'], df_combined['power_consumption'],
           alpha=0.3, s=10, c='red')
ax.set_xlabel('Environmental Stress Factor')
ax.set_ylabel('Power Consumption (W)')
ax.set_title('Environmental Stress vs Power', fontweight='bold')

# 3. Wind Speed Squared Distribution
ax = axes[0, 2]
ax.hist(df_combined['wind_speed_squared'], bins=40, color='orange', edgecolor='white')
ax.set_xlabel('Wind Speed²')
ax.set_ylabel('Frequency')
ax.set_title('Wind Speed² Distribution', fontweight='bold')

# 4. Power Loading vs Range
ax = axes[1, 0]
ax.scatter(df_combined['power_loading'], df_combined['max_range'],
           alpha=0.3, s=10, c='green')
ax.set_xlabel('Power Loading (W/kg)')
ax.set_ylabel('Max Range (km)')
ax.set_title('Power Loading vs Range', fontweight='bold')

# 5. SoC-Power Ratio vs Range
ax = axes[1, 1]
ax.scatter(df_combined['soc_power_ratio'], df_combined['max_range'],
           alpha=0.3, s=10, c='purple')
ax.set_xlabel('SoC/Power Ratio')
ax.set_ylabel('Max Range (km)')
ax.set_title('SoC-Power Ratio vs Range', fontweight='bold')

# 6. Temperature Efficiency Distribution
ax = axes[1, 2]
ax.hist(df_combined['temp_efficiency'], bins=40, color='teal', edgecolor='white')
ax.set_xlabel('Temperature Efficiency Factor')
ax.set_ylabel('Frequency')
ax.set_title('Temperature Efficiency Distribution', fontweight='bold')

plt.suptitle('Engineered Features Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 Step 4: Feature Correlation with Target

In [ ]:
def pearson_correlation(x, y):
    """Calculate Pearson correlation from scratch."""
    x = np.array(x)
    y = np.array(y)
    x_mean, y_mean = np.mean(x), np.mean(y)
    numerator = np.sum((x - x_mean) * (y - y_mean))
    denominator = np.sqrt(np.sum((x - x_mean)**2) * np.sum((y - y_mean)**2))
    return numerator / denominator if denominator != 0 else 0


# Calculate correlations with target
target = df_combined['max_range']
correlations = {}

for col in engineered_features.columns:
    correlations[col] = pearson_correlation(df_combined[col], target)

# Sort by absolute correlation
sorted_corr = sorted(correlations.items(), key=lambda x: abs(x[1]), reverse=True)

print("=" * 60)
print("  ENGINEERED FEATURES CORRELATION WITH MAX_RANGE")
print("=" * 60)
for name, corr in sorted_corr:
    strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
    bar = '█' * int(abs(corr) * 20)
    sign = '+' if corr > 0 else '-'
    print(f"  {name:30s}: {sign}{abs(corr):.3f} {bar} ({strength})")

In [ ]:
# Correlation bar chart
plt.figure(figsize=(12, 8))
names = [x[0] for x in sorted_corr]
values = [x[1] for x in sorted_corr]
colors = ['green' if v > 0 else 'red' for v in values]

plt.barh(range(len(names)), values, color=colors, edgecolor='white')
plt.yticks(range(len(names)), names)
plt.xlabel('Correlation with Max Range', fontsize=12)
plt.title('Engineered Features: Correlation with Target', fontsize=14, fontweight='bold')
plt.axvline(x=0, color='black', linewidth=0.5)
plt.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axvline(x=-0.5, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('../reports/figures/engineered_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 🎯 Step 5: Feature Selection

In [ ]:
class FeatureSelector:
    """
    Feature selection using correlation analysis and variance threshold.
    """
    
    def __init__(self, correlation_threshold=0.1, variance_threshold=0.01):
        self.correlation_threshold = correlation_threshold
        self.variance_threshold = variance_threshold
        self.selected_features = None
        self.removed_features = None
    
    def fit(self, X, y, feature_names):
        """
        Select features based on correlation and variance.
        """
        X = np.array(X)
        y = np.array(y)
        
        self.selected_features = []
        self.removed_features = {'low_variance': [], 'low_correlation': []}
        
        for i, name in enumerate(feature_names):
            # Check variance
            variance = np.var(X[:, i])
            if variance < self.variance_threshold:
                self.removed_features['low_variance'].append(name)
                continue
            
            # Check correlation with target
            corr = abs(pearson_correlation(X[:, i], y))
            if corr < self.correlation_threshold:
                self.removed_features['low_correlation'].append(name)
                continue
            
            self.selected_features.append((name, i, corr))
        
        # Sort by correlation
        self.selected_features.sort(key=lambda x: x[2], reverse=True)
        
        return self
    
    def transform(self, X):
        """
        Return only selected features.
        """
        X = np.array(X)
        indices = [f[1] for f in self.selected_features]
        return X[:, indices]
    
    def get_selected_names(self):
        """Return names of selected features."""
        return [f[0] for f in self.selected_features]
    
    def report(self):
        """Print selection report."""
        print("=" * 50)
        print("         FEATURE SELECTION REPORT")
        print("=" * 50)
        print(f"\nSelected Features: {len(self.selected_features)}")
        for name, idx, corr in self.selected_features:
            print(f"  ✓ {name} (corr: {corr:.3f})")
        
        print(f"\nRemoved - Low Variance: {len(self.removed_features['low_variance'])}")
        for name in self.removed_features['low_variance']:
            print(f"  ✗ {name}")
        
        print(f"\nRemoved - Low Correlation: {len(self.removed_features['low_correlation'])}")
        for name in self.removed_features['low_correlation']:
            print(f"  ✗ {name}")


# Apply feature selection on engineered features
selector = FeatureSelector(correlation_threshold=0.15, variance_threshold=0.01)
selector.fit(engineered_features.values, df['max_range'].values, engineered_features.columns.tolist())
selector.report()

## 💾 Step 6: Create Final Feature Set & Save

In [ ]:
# Define final feature set
# Combine original important features + best engineered features

original_numeric_features = [
    'battery_soc', 'voltage', 'current', 'battery_temperature',
    'wind_speed', 'dust_level', 'ambient_temperature',
    'altitude', 'payload_weight', 'flight_speed', 'power_consumption'
]

# Select top engineered features (correlation > 0.2)
top_engineered = [f[0] for f in selector.selected_features if f[2] > 0.2]

print("Final Feature Set:")
print("=" * 50)
print(f"\nOriginal Features ({len(original_numeric_features)}):")
for f in original_numeric_features:
    print(f"  • {f}")

print(f"\nTop Engineered Features ({len(top_engineered)}):")
for f in top_engineered:
    print(f"  • {f}")

print(f"\nTotal features: {len(original_numeric_features) + len(top_engineered)}")

In [ ]:
# Create final dataset with engineered features
final_df = df.copy()

# Add engineered features
for col in top_engineered:
    final_df[col] = engineered_features[col].values

# Save final feature-engineered dataset
output_path = '../data/processed/uav_telemetry_features.csv'
final_df.to_csv(output_path, index=False)

print(f"\n✅ Feature-engineered dataset saved to: {output_path}")
print(f"   Shape: {final_df.shape}")
print(f"\nPreview:")
final_df.head()

In [ ]:
# Save feature engineering configuration
feature_config = {
    'original_features': original_numeric_features,
    'engineered_features': top_engineered,
    'categorical_features': ['terrain_type'],
    'target_regression': 'max_range',
    'target_classification': 'mission_success',
    'feature_engineering_formulas': {
        'available_energy_wh': 'battery_soc / 100 * 222',
        'wind_speed_squared': 'wind_speed ** 2',
        'environmental_stress': 'wind_power_factor * dust_impact',
        'total_weight': '3.5 + payload_weight',
        'power_loading': 'power_consumption / total_weight',
        'soc_power_ratio': 'battery_soc / (power_consumption / 100 + 1)'
    }
}

with open('../models/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)

print("\n✅ Feature configuration saved to: ../models/feature_config.json")

---

## 📚 Learning Resources

### Blogs & Articles
1. **Feature Engineering**: [A Complete Guide](https://www.kaggle.com/learn/feature-engineering)
2. **Polynomial Features**: [When and Why to Use](https://machinelearningmastery.com/polynomial-features-transforms-for-machine-learning/)
3. **Domain Knowledge in ML**: [The Art of Feature Engineering](https://towardsdatascience.com/the-art-of-feature-engineering-d79e6f66a6aa)

### YouTube Videos
1. **Feature Engineering Tutorial**: [Complete Guide](https://www.youtube.com/watch?v=6WDFfaYtN6s)
2. **Polynomial Regression**: [Visual Explanation](https://www.youtube.com/watch?v=QptI-vDle8Y)

### Research Papers
1. Zheng, A. & Casari, A. (2018). "Feature Engineering for Machine Learning"
2. Zinkevich, M. - Rules #16-#22 on Feature Engineering

---

## ✅ Task 4 Complete!

### What We Accomplished:
- ✅ Built polynomial feature generator from scratch
- ✅ Created physics-based features (energy, efficiency, environmental)
- ✅ Implemented interaction features
- ✅ Applied feature selection based on correlation
- ✅ Saved enhanced dataset and configuration

### Key Engineered Features:
| Feature | Type | Correlation |
|---------|------|-------------|
| soc_power_ratio | Interaction | Strong |
| available_energy_wh | Physics | Strong |
| theoretical_flight_time | Physics | Strong |
| wind_speed_squared | Polynomial | Moderate |
| environmental_stress | Interaction | Moderate |

### Next Task: Feature Importance Visualization & Linear Regression
We'll build our first ML model from scratch!

---